# Lab 1.2, Build 2 — Build and update the pipeline

**Before you start:** select **Cell > Run All** to initialize the harness. Then work through the cells marked `# ── YOUR WORK ──`.

**Two parts:**
- **Build A:** Write the semantic retrieval query and inject context into Tina's prompt.
- **Build B:** Update `policy-003` in Elasticsearch and measure propagation latency.

**Acceptance:** precision@3 ≥ 0.80 on 5 held-out queries; updated value visible after PUT.

In [ ]:
# Harness setup — run once
import sys, os, json, pathlib, time
sys.path.insert(0, '/opt/ara/lib')

env_path = pathlib.Path('/home/elastic/env')
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

from tina.client import llm_client, es_client, model_fast
client = llm_client()
es = es_client()
FAST = model_fast()
TRACES = pathlib.Path('/home/elastic/.traces')
TRACES.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = (
    'You are Tina, Cortex Bank and Trust\'s compliance assistant. '
    'Answer compliance questions concisely and cite the policy source.'
)

# Load dev queries
dev_queries = [json.loads(l) for l in
               pathlib.Path('/home/elastic/dev-sets/dev-queries.jsonl').read_text().splitlines()
               if l.strip()]
print(f'Harness ready. {len(dev_queries)} dev queries loaded.')

---
## Build A — Retrieval pipeline
Write `retrieve_and_answer(question)` using a semantic search over `cortex-policies`, inject top-3 context into the prompt, and call the LLM.

In [ ]:
# ── YOUR WORK ── Build A: retrieval pipeline ───────────────────────────────────
# Write retrieve_and_answer(question) that:
#  1. Runs a semantic search on cortex-policies (field: body_semantic, k=3).
#  2. Filters to effective_date <= now (see Brief slide 9).
#  3. Injects context into the prompt.
#  4. Calls the LLM and returns the answer.
#
# Template:
# resp = es.search(
#     index='cortex-policies',
#     body={
#         'query': {'semantic': {'field': 'body_semantic', 'query': question}},
#         # optional: add a filter for effective_date
#     },
#     size=3,
# )
# hits = resp['hits']['hits']
# context = '\n\n'.join(h['_source']['body'] for h in hits)
# retrieved_ids = [h['_id'] for h in hits]
#
# prompt = f"""Use the following Cortex Bank policy excerpts to answer:\n\n{context}\n\nQuestion: {question}"""
# resp2 = client.chat.completions.create(
#     model=FAST, messages=[{'role':'system','content':SYSTEM_PROMPT},
#                           {'role':'user','content':prompt}], temperature=0)
# return resp2.choices[0].message.content

def retrieve_and_answer(question: str) -> dict:
    """Return dict with answer and retrieved_ids."""
    # ── YOUR CODE HERE ──
    return {'answer': '', 'retrieved_ids': []}  # replace

In [ ]:
# Run pipeline on dev queries
dev_results = []
for q in dev_queries:
    result = retrieve_and_answer(q['query_text'])
    dev_results.append({'query_id': q['query_id'], **result})
    print(f"Q: {q['query_text'][:60]}")
    print(f"   Retrieved: {result.get('retrieved_ids', [])}")
    print(f"   Answer: {result.get('answer','')[:100]}")
    print()

---
## Build B — Update and measure propagation
Update `policy-003`'s CTR threshold in Elasticsearch. Measure how long it takes the new value to appear in a retrieval result.

In [ ]:
# ── YOUR WORK ── Build B: update policy-003 and measure propagation ────────────
# 1. Choose a new CTR threshold value (must be different from 10000).
# 2. Update the policy-003-0 document in cortex-policies via es.update().
# 3. Start a timer and repeatedly query for the CTR threshold until the new value appears.
# 4. Record the propagation latency.
#
# Template:
# NEW_THRESHOLD = "12500"  # your chosen value
# t0 = time.perf_counter()
# es.update(index='cortex-policies', id='policy-003-0',
#           body={'doc': {'body': f'The CTR threshold is ${NEW_THRESHOLD}.',
#                         'body_semantic': f'CTR threshold is ${NEW_THRESHOLD}.'}})
# # Poll until the new value appears
# for attempt in range(30):
#     result = retrieve_and_answer('What is the CTR threshold?')
#     if NEW_THRESHOLD in result['answer']:
#         break
#     time.sleep(1)
# propagation_latency_ms = (time.perf_counter() - t0) * 1000

NEW_THRESHOLD = '12500'    # ← change if you want
propagation_latency_ms = -1  # ← replace with your measurement
print(f'Propagation latency: {propagation_latency_ms:.0f} ms')

In [ ]:
# Record pipeline results — run after both Build A and Build B are complete
assert dev_results, 'Run the dev queries first.'
(TRACES / 'pipeline-results.json').write_text(json.dumps({
    'precision_at_3': None,  # computed by the check script on held-out set
    'updated_threshold_value': NEW_THRESHOLD,
    'propagation_latency_ms': propagation_latency_ms,
    'held_out_count': 5,
    'timestamp': time.time(),
}, indent=2))
print('Results recorded. Select Check in the sidebar.')